# 02 Forecasting

Baseline forecasting workflow for electricity price and carbon intensity.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/saab/Documents/DSBA/Causal Energy Inelligence/causal-energy-intelligence


In [10]:
from src.models.baseline_price import run_price_baselines

result = run_price_baselines(
    data_path=PROJECT_ROOT / "data/processed/modeling_price_features.csv",
    metrics_path=PROJECT_ROOT / "reports/metrics/price_baseline_metrics.json",
    predictions_path=PROJECT_ROOT / "reports/predictions/price_baseline_predictions.csv",
    diagnostics_path=PROJECT_ROOT / "reports/metrics/price_baseline_error_diagnostics.csv",
    artifacts_dir=PROJECT_ROOT / "models",
)

result["summary"]

/Users/saab/Documents/DSBA/Causal Energy Inelligence/causal-energy-intelligence/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/saab/Documents/DSBA/Causal Energy Inelligence/causal-energy-intelligence/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/saab/Documents/DSBA/Causal Energy Inelligence/causal-energy-intelligence/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/saab/Documents/DSBA/Causal Energy Inelligence/causal-energy-intelligence/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMR

[{'model': 'lightgbm',
  'mae': 8.109578410415653,
  'rmse': 12.933546072668946,
  'smape': 0.35100666219777005,
  'directional_accuracy': 0.7661658098237792},
 {'model': 'hist_gradient_boosting',
  'mae': 8.159414585388886,
  'rmse': 13.012815333773341,
  'smape': 0.3497124433628979,
  'directional_accuracy': 0.7577249461433927},
 {'model': 'xgboost',
  'mae': 8.52180168856487,
  'rmse': 13.0727677026997,
  'smape': 0.36025473197810287,
  'directional_accuracy': 0.7635201583831369},
 {'model': 'random_forest',
  'mae': 8.75348071399615,
  'rmse': 13.76123642111111,
  'smape': 0.3591798067467527,
  'directional_accuracy': 0.7475343159521566},
 {'model': 'ridge',
  'mae': 12.362340691586216,
  'rmse': 16.93246124815819,
  'smape': 0.4466071169369732,
  'directional_accuracy': 0.7400556227633218},
 {'model': 'naive_lag_24h',
  'mae': 25.480637272360607,
  'rmse': 35.838426110950635,
  'smape': 0.6610668526824738,
  'directional_accuracy': 0.7819454270808974}]

In [11]:
import pandas as pd

metrics_df = pd.DataFrame(result["metrics"])
summary_df = pd.DataFrame(result["summary"])

summary_df

,model,mae,rmse,smape,directional_accuracy
0,lightgbm,8.109578,12.933546,0.351007,0.766166
1,hist_gradient_boosting,8.159415,13.012815,0.349712,0.757725
2,xgboost,8.521802,13.072768,0.360255,0.763520
3,random_forest,8.753481,13.761236,0.359180,0.747534
4,ridge,12.362341,16.932461,0.446607,0.740056
5,naive_lag_24h,25.480637,35.838426,0.661067,0.781945


In [12]:
metrics_df

,window,model,train_rows,test_rows,mae,rmse,smape,directional_accuracy
0,"{'name': 'wf_2026_01', 'train_start': '2023-01...",naive_lag_24h,23757,744,19.373327,26.410783,0.218060,0.804845
1,"{'name': 'wf_2026_01', 'train_start': '2023-01...",ridge,23757,744,6.892479,9.499986,0.078225,0.773890
2,"{'name': 'wf_2026_01', 'train_start': '2023-01...",random_forest,23757,744,5.505795,7.958581,0.061472,0.787349
3,"{'name': 'wf_2026_01', 'train_start': '2023-01...",hist_gradient_boosting,23757,744,5.112325,7.150368,0.058002,0.786003
4,"{'name': 'wf_2026_01', 'train_start': '2023-01...",lightgbm,23757,744,4.989005,6.967805,0.056874,0.800808
5,"{'name': 'wf_2026_01', 'train_start': '2023-01...",xgboost,23757,744,5.154679,7.202442,0.058689,0.803499
6,"{'name': 'wf_2026_02', 'train_start': '2023-01...",naive_lag_24h,24501,672,20.843188,29.704006,0.757013,0.752608
7,"{'name': 'wf_2026_02', 'train_start': '2023-01...",ridge,24501,672,9.556366,12.912819,0.497080,0.722802
8,"{'name': 'wf_2026_02', 'train_start': '2023-01...",random_forest,24501,672,8.020359,11.598272,0.454611,0.707899
9,"{'name': 'wf_2026_02', 'train_start': '2023-01...",hist_gradient_boosting,24501,672,7.664899,10.935474,0.438629,0.730253


In [13]:
import pandas as pd

predictions = pd.read_csv(
    PROJECT_ROOT / "reports/predictions/price_baseline_predictions.csv",
    parse_dates=["timestamp_utc"],
)

In [14]:
diagnostics = pd.read_csv(
    PROJECT_ROOT / "reports/metrics/price_baseline_error_diagnostics.csv"
)

diagnostics.head()

,model,hour,rows,mae,rmse,diagnostic_group,day_of_week,price_regime
0,hist_gradient_boosting,0.0,160,5.449819,7.405707,hour,NaN,NaN
1,hist_gradient_boosting,1.0,160,5.592528,8.480669,hour,NaN,NaN
2,hist_gradient_boosting,2.0,160,5.527262,8.131624,hour,NaN,NaN
3,hist_gradient_boosting,3.0,160,5.670892,7.823507,hour,NaN,NaN
4,hist_gradient_boosting,4.0,160,7.405140,10.378482,hour,NaN,NaN
